# Car Spotter dashboard

The Streamlit dashboard, in one notebook. Each cell below writes one file of the app (`dashboard/` and the theme in `.streamlit/`) with `%%writefile`, and the text above it explains what that file does.

**This notebook is where the dashboard is edited**: change a cell, run it, and refresh the browser (Streamlit reloads a page when its file changes). Run all the cells once to write the whole app, then the last section checks every page and starts the app.

The pages only read what the notebooks saved to `models/` (`car_brand_predictor.ipynb`, `car_model_predictor.ipynb`, `car_model_finetune.ipynb`); nothing is trained here.

Pages: **Home**, **Data**, **Brand model**, **Car models**, **Identify**.

In [1]:
from pathlib import Path

# Streamlit is started from the project root, so the files go there
for folder in ["dashboard", ".streamlit"]:
    Path(folder).mkdir(exist_ok=True)

## 1. Theme

Streamlit reads its colours and font from `.streamlit/config.toml` in the folder it's started from (the project root). The palette is the Fair Car Price one (graphite panels, warm white text, Space Grotesk) with **Spotify green** in place of the gold.

In [2]:
%%writefile .streamlit/config.toml
# Car Spotter dashboard theme: Graphite & Green (dark), Space Grotesk. Same family as the Fair Car Price dashboard.
[theme]
base = "dark"
primaryColor = "#1db954"
backgroundColor = "#0c0c0e"
secondaryBackgroundColor = "#161618"
textColor = "#f2efe8"
borderColor = "#2a2a2e"
linkColor = "#1db954"
font = "'Space Grotesk':https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300..700&display=swap, sans-serif"
chartCategoricalColors = ["#1db954", "#5a8fe0", "#e05a3a", "#e0a83a"]
codeTextColor = "#f2efe8"
codeBackgroundColor = "#2a2a2e"

Overwriting .streamlit/config.toml


## 2. Logo

The car icon (Material Symbols' `directions_car`) and the name, in green. `st.logo` shows it in the top bar of every page.

In [3]:
%%writefile dashboard/logo.svg
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 520 64"><path fill="#1db954" transform="translate(0 2) scale(2.5)" d="M18.92 6.01C18.72 5.42 18.16 5 17.5 5h-11c-.66 0-1.21.42-1.42 1.01L3 12v8c0 .55.45 1 1 1h1c.55 0 1-.45 1-1v-1h12v1c0 .55.45 1 1 1h1c.55 0 1-.45 1-1v-8l-2.08-5.99zM6.5 16c-.83 0-1.5-.67-1.5-1.5S5.67 13 6.5 13s1.5.67 1.5 1.5S7.33 16 6.5 16zm11 0c-.83 0-1.5-.67-1.5-1.5s.67-1.5 1.5-1.5 1.5.67 1.5 1.5-.67 1.5-1.5 1.5zM5 11l1.5-4.5h11L19 11H5z"/><text x="78" y="47" fill="#1db954" font-family="Helvetica, Arial, sans-serif" font-weight="700" font-size="44" letter-spacing="-0.5">Car Spotter</text></svg>

Overwriting dashboard/logo.svg


## 3. Shared pieces: `common.py`

Used by every page:
- the **palette** (the same colours as the theme, for the charts)
- a **Plotly template** so every chart has the dark panel background, the font and the green-first colour order
- `panel(title)`: a rounded box with a title, for grouping a chart with its caption
- `chart(fig)`: shows a Plotly figure with the template (and stops Streamlit restyling it)
- `load(path, notebook)`: reads a CSV the notebooks saved to `models/`, cached, and stops the page with a hint if that notebook hasn't been run yet

In [4]:
%%writefile dashboard/common.py
"""Shared by every dashboard page: the palette, the Plotly chart style, panels, and loading the saved results.

The numbers come from the notebooks (car_brand_predictor, car_model_predictor, car_model_finetune), which save
their results to models/; the pages only read them.
"""

import re
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

MODELS_DIR = Path("models")
DATA_DIR = Path("data")

# palette: Graphite & Green (dark), same family as the Fair Car Price dashboard with green for gold
BG = "#0c0c0e"        # page
PANEL = "#161618"     # panels and cards
LINE = "#2a2a2e"      # borders, gridlines, faint marks
TEXT = "#f2efe8"      # warm white
SUBTEXT = "#a09c94"   # secondary text
GREEN = "#1db954"     # the accent: main series, buttons, logo
BLUE = "#5a8fe0"      # second series
RED = "#e05a3a"       # bad, mistakes
AMBER = "#e0a83a"     # fourth series
GRAPHITE = "#6b6b73"  # neutral series
GREY = "#4a4a50"      # reference marks
FONT = "Space Grotesk, sans-serif"

TEMPLATE = go.layout.Template(layout=dict(
    font=dict(family=FONT, color=TEXT, size=13),
    paper_bgcolor=PANEL,
    plot_bgcolor=PANEL,
    colorway=[GREEN, BLUE, RED, AMBER],
    xaxis=dict(showgrid=False, zeroline=False, linecolor=GRAPHITE, tickcolor=GRAPHITE, automargin=True),
    yaxis=dict(gridcolor=LINE, zeroline=False, automargin=True),
    legend=dict(bgcolor="rgba(0,0,0,0)"),
    margin=dict(l=10, r=10, t=10, b=10),
    hoverlabel=dict(font_family=FONT),
))

STYLE = f"""<style>
[class*="st-key-panel_"] {{ background: {PANEL}; border: 1px solid {LINE}; border-radius: 1rem; padding: 1.25rem 1.5rem; }}
[data-testid="stMetric"] {{ background: {PANEL}; }}
[data-testid="stAppDeployButton"], [data-testid="stMainMenu"], [data-testid="stStatusWidget"],
[data-testid="stDecoration"] {{ display: none; }}
[data-testid^="stBaseButton-primary"], [data-testid^="stBaseButton-primary"] p {{ color: {BG}; font-weight: 700; }}
</style>"""


def style():
    st.html(STYLE)


def panel(title, key=None):
    """A rounded panel box with the title inside: `with panel("Accuracy per brand"): ...`"""
    box = st.container(key="panel_" + re.sub(r"\W+", "_", (key or title).lower()))
    box.markdown(f"**{title}**")
    return box


def chart(fig):
    # theme=None stops Streamlit restyling the chart; the backgrounds are set on the figure itself
    fig.update_layout(template=TEMPLATE, paper_bgcolor=PANEL, plot_bgcolor=PANEL)
    st.plotly_chart(fig, theme=None)


@st.cache_data
def _read(path, modified, usecols):
    return pd.read_csv(path, usecols=usecols)


def load(path, notebook, usecols=None):
    """A saved CSV; stops the page with a hint when the notebook that writes it hasn't been run yet."""
    path = Path(path)
    if not path.exists():
        st.warning(f"`{path}` not found: run `{notebook}` first.")
        st.stop()
    return _read(path, path.stat().st_mtime, usecols)  # the timestamp re-reads the file after a re-run


def pct(x, digits=1):
    return f"{x:.{digits}%}"

Overwriting dashboard/common.py


## 4. The recognition itself: `recognition.py`

Everything the **Identify** page needs, and nothing about how it looks:
- `crop_car`: YOLO11-seg finds the biggest car, paints the background white and centres it on a 300×300 square, like the training photos
- `predict_brand`: the make, from the front photo, with the brand model (`car_brand_predictor.ipynb`)
- `predict_car_model`: for the makes with at least 5% probability (up to 3), their car-model classifier on every photo, averaged. It uses the fine-tuned classifier (`car_model_finetune.ipynb`) when one exists, otherwise the classifier on features (`car_model_predictor.ipynb`). Each (make, model) scores P(make) × P(model | make).

The models are loaded once and cached with `st.cache_resource`; the crops are cached per photo with `st.cache_data`.

In [5]:
%%writefile dashboard/recognition.py
"""The two-step recognition used by the Identify page.

Step 1, brand: the front photo goes to the brand model (car_brand_predictor.ipynb).
Step 2, model: every photo goes to the car-model classifiers of the likely brands (car_model_predictor.ipynb,
or the fine-tuned ones from car_model_finetune.ipynb when they exist), averaged over the photos.
"""
import io
import json
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image, ImageOps

from common import DATA_DIR, MODELS_DIR

YOLO_PATH = MODELS_DIR / "yolo11m-seg.pt"
BRAND_MODEL_PATH = MODELS_DIR / "efficientnet_b0.keras"
HEADS_DIR = MODELS_DIR / "model_heads"
FINETUNED_DIR = MODELS_DIR / "model_finetuned"

CAR_CLASSES = [2, 7]      # COCO: car, truck (SUVs and pickups are often detected as "truck")
CROP_SIZE = 300           # the training photos are 300×300
MODEL_SIZE = 224          # EfficientNetB0 takes 224×224
MIN_BRAND_PROBABILITY = 0.05
MAX_CANDIDATE_BRANDS = 3


@st.cache_resource(show_spinner="Loading the car detector (first time only)...")
def load_yolo():
    from ultralytics import YOLO
    return YOLO(str(YOLO_PATH))


@st.cache_resource(show_spinner="Loading the brand model (first time only)...")
def load_brand_model():
    import keras
    return keras.models.load_model(BRAND_MODEL_PATH)


@st.cache_resource
def load_feature_extractor():
    """The brand model's EfficientNet + average pooling: the features the car-model classifiers were trained on."""
    import keras
    from keras import layers
    base = load_brand_model().get_layer("efficientnetb0")
    inputs = keras.Input(shape=(MODEL_SIZE, MODEL_SIZE, 3))
    return keras.Model(inputs, layers.GlobalAveragePooling2D()(base(inputs, training=False)))


def is_finetuned(brand):
    return (FINETUNED_DIR / f"{brand}_done.txt").exists() and (FINETUNED_DIR / f"{brand}.keras").exists()


@st.cache_resource(max_entries=12)
def load_car_model_classifier(brand, finetuned):
    """(model, classes): the fine-tuned model (takes images) or the classifier on features."""
    import keras
    classes = json.loads((HEADS_DIR / f"{brand}_classes.json").read_text())
    path = FINETUNED_DIR / f"{brand}.keras" if finetuned else HEADS_DIR / f"{brand}.keras"
    return keras.models.load_model(path), classes


@st.cache_data
def brand_names():
    """Brands in label order: alphabetical, exactly as during training."""
    return sorted(pd.read_csv(DATA_DIR / "index.csv", usecols=["brand"])["brand"].unique())


def car_models_available():
    return all((HEADS_DIR / f"{b}.keras").exists() for b in brand_names())


@st.cache_data(max_entries=64, show_spinner=False)
def crop_car(photo_bytes):
    """Cut out the biggest car, white background, centred on a square: the style of the training photos.
    Returns (photo, crop, detection confidence); crop and confidence are None when no car is found."""
    photo = ImageOps.exif_transpose(Image.open(io.BytesIO(photo_bytes))).convert("RGB")
    pixels = np.array(photo)
    result = load_yolo()(pixels, classes=CAR_CLASSES, retina_masks=True, verbose=False)[0]
    if result.masks is None:
        return photo, None, None
    masks = result.masks.data.cpu().numpy() > 0.5
    biggest = masks.sum(axis=(1, 2)).argmax()
    white = np.where(masks[biggest][..., None], pixels, 255).astype(np.uint8)
    # crop to the detection box: the mask can have stray pixels far from the car
    x1, y1, x2, y2 = result.boxes.xyxy[biggest].cpu().numpy().round().astype(int)
    car = white[max(y1, 0):y2, max(x1, 0):x2]
    side = max(car.shape[:2])
    square = np.full((side, side, 3), 255, np.uint8)
    top, left = (side - car.shape[0]) // 2, (side - car.shape[1]) // 2
    square[top:top + car.shape[0], left:left + car.shape[1]] = car
    crop = Image.fromarray(square).resize((CROP_SIZE, CROP_SIZE), Image.LANCZOS)
    return photo, crop, float(result.boxes.conf[biggest])


def _inputs(crops):
    import tensorflow as tf
    return tf.image.resize(np.stack([np.array(c, dtype="float32") for c in crops]), (MODEL_SIZE, MODEL_SIZE))


def predict_brand(front_crop):
    """[(brand, probability)], most likely first."""
    import tensorflow as tf
    probabilities = tf.nn.softmax(load_brand_model()(_inputs([front_crop]), training=False)[0]).numpy()
    names = brand_names()
    return [(names[i], float(probabilities[i])) for i in probabilities.argsort()[::-1]]


def predict_car_model(brand_ranking, crops):
    """Run the car-model classifiers of the likely brands on every photo.

    Returns (ranking, per_photo, finetuned):
    - ranking: [(brand, model, score)] with score = P(brand) × P(model | brand), best first
    - per_photo: {brand: [(model, probability)] one per photo}
    - finetuned: {brand: True if its fine-tuned classifier was used}
    """
    import tensorflow as tf
    candidates = [(b, p) for b, p in brand_ranking[:MAX_CANDIDATE_BRANDS] if p >= MIN_BRAND_PROBABILITY] or brand_ranking[:1]
    x = _inputs(crops)
    features = None
    ranking, per_photo, finetuned = [], {}, {}
    for brand, p_brand in candidates:
        finetuned[brand] = is_finetuned(brand)
        classifier, classes = load_car_model_classifier(brand, finetuned[brand])
        if finetuned[brand]:
            scores = classifier(x, training=False)
        else:
            if features is None:
                features = load_feature_extractor()(x, training=False)
            scores = classifier(features, training=False)
        photo_p = tf.nn.softmax(scores).numpy()
        per_photo[brand] = [(classes[p.argmax()], float(p.max())) for p in photo_p]
        car_p = photo_p.mean(axis=0)
        ranking += [(brand, classes[i], p_brand * float(car_p[i])) for i in range(len(classes))]
    ranking.sort(key=lambda r: r[2], reverse=True)
    return ranking, per_photo, finetuned

Overwriting dashboard/recognition.py


## 5. The app: `app.py`

The entry point: page title and icon, the logo, the shared style, and the navigation bar with the five pages at the top.

In [6]:
%%writefile dashboard/app.py
"""Car Spotter dashboard. Run from the project root: .venv/bin/streamlit run dashboard/app.py"""

from pathlib import Path

import streamlit as st

from common import style

HERE = Path(__file__).parent

st.set_page_config(page_title="Car Spotter", page_icon=":material/directions_car:", layout="wide")
st.logo(str(HERE / "logo.svg"), size="large")
style()

pages = [
    st.Page("home.py", title="Home", default=True),
    st.Page("data.py", title="Data"),
    st.Page("brand_model.py", title="Brand model"),
    st.Page("car_models.py", title="Car models"),
    st.Page("identify.py", title="Identify"),
]
st.navigation(pages, position="top").run()

Overwriting dashboard/app.py


## 6. Home

The front page: the catchy question, how many makes and models the tool knows, a button to the Identify page, three headline numbers and "How it works" in three steps. The numbers are read from the saved results, so they update when a notebook is re-run.

In [7]:
%%writefile dashboard/home.py
import json

import streamlit as st

from common import GREEN, MODELS_DIR, SUBTEXT, TEXT, load, pct

brand_test = load(MODELS_DIR / "test_predictions.csv", "car_brand_predictor.ipynb", usecols=["brand", "correct"])
pipeline = load(MODELS_DIR / "model_heads" / "pipeline_test.csv", "car_model_predictor.ipynb")
makes = brand_test["brand"].nunique()
models = sum(len(json.loads(p.read_text())) for p in (MODELS_DIR / "model_heads").glob("*_classes.json"))
photos = json.loads((MODELS_DIR / "features" / "features_info.json").read_text())["done"]

# ---------- hero ----------
st.markdown(
    f"<div style='max-width:50rem; padding:3rem 0 1.5rem'>"
    f"<div style='font-size:3.2rem; font-weight:700; line-height:1.1; color:{TEXT}'>"
    f"Don't know what you're driving?</div>"
    f"<p style='font-size:1.25rem; line-height:1.6; color:{SUBTEXT}; margin-top:1.5rem'>"
    f"There are <b style='color:{TEXT}'>{makes} makes</b> and <b style='color:{TEXT}'>{models} models</b> "
    f"in this garage, and plenty of them look alike from across the street. Is that an A4 or an A5? "
    f"A 3 Series or a 4 Series?</p>"
    f"<p style='font-size:1.25rem; line-height:1.6; color:{TEXT}'>"
    f"Show me a few photos and I'll <b style='color:{GREEN}'>figure out what you're driving</b>: "
    f"the make from the front, the model from every angle.</p></div>",
    unsafe_allow_html=True,
)
if st.button("Identify a car →", type="primary"):
    st.switch_page("identify.py")
st.caption("Trained on photos from UK car adverts: the DVM-CAR dataset (Huang et al., 2022).")

# ---------- numbers ----------
st.write("")
cards = st.columns(3)
cards[0].metric("Car photos it learned from", f"{photos:,}", border=True,
                help="Photos of the 33 makes from every angle, after removing broken photos and duplicates: "
                     "see the Data page.")
cards[1].metric("Make right", pct(brand_test["correct"].mean()), border=True,
                help="Share of test cars whose make is right, from one front photo the model never saw: "
                     "see the Brand model page.")
cards[2].metric("Make and model right", pct(pipeline["both correct"].mean()), border=True,
                help=f"The whole tool on {len(pipeline):,} test cars it never saw: make from the front photo, "
                     "model from all the car's photos. See the Car models page.")

# ---------- how it works ----------
st.subheader("How it works")
STEPS = [
    ("Upload your photos", "One or more photos of the same car, from any side. Mark the one that shows the front."),
    ("Get the make", f"The front photo goes to a model trained on 60,000 car fronts. It picks one of {makes} makes."),
    ("Get the model", f"Every photo then goes to that make's own classifier, which knows its models from every "
                      f"angle. More photos, surer answer."),
]
for column, (number, (title, text)) in zip(st.columns(3), enumerate(STEPS, start=1)):
    with column.container(border=True):
        st.markdown(f"<div style='font-size:2rem; font-weight:700; color:{GREEN}'>{number}</div>"
                    f"<div style='font-weight:700; margin:0.25rem 0'>{title}</div>"
                    f"<div style='color:{SUBTEXT}'>{text}</div>", unsafe_allow_html=True)

st.write("")
st.divider()
st.caption("**Note:** the tool only knows the makes and models it was trained on, mostly UK cars from 2000–2021. "
           "Anything else is reported as the closest car it knows.")

Overwriting dashboard/home.py


## 7. Data

Where the photos come from and how they were cleaned:
- two funnels, fronts (for the make) and all angles (for the model); the counts are the ones the notebooks printed while cleaning
- one car from all 8 angles
- photos per make and per angle, and the split by advert

In [8]:
%%writefile dashboard/data.py
import plotly.graph_objects as go
import streamlit as st

from common import BLUE, DATA_DIR, GREEN, LINE, MODELS_DIR, RED, TEXT, chart, load, panel

# DVM-CAR's predicted viewpoint, in degrees around the car
ANGLES = {0: "front (0°)", 45: "45°", 90: "side (90°)", 135: "135°", 180: "rear (180°)",
          225: "225°", 270: "side (270°)", 315: "315°"}

# counts printed by the notebooks while cleaning (section 1 of each)
FRONTS_FUNNEL = [  # car_brand_predictor.ipynb
    ("In DVM-CAR", 61_827),
    ("Under 2 makes", -125),
    ("Duplicates", -500),
    ("Rare makes", -1_076),
]
ALL_ANGLES_FUNNEL = [  # car_model_predictor.ipynb
    ("In DVM-CAR", 1_451_784),
    ("Filtered", -78_460),
    ("2 labels", -4_795),
    ("Duplicates", -16_710),
]


@st.cache_data
def photo_counts(modified):
    photos = load(MODELS_DIR / "features" / "photo_index.csv", "car_model_predictor.ipynb",
                  usecols=["path", "brand", "model", "angle", "advert_id", "split"])
    per_brand = photos.groupby("brand").agg(photos=("path", "size"), models=("model", "nunique")).sort_values("photos")
    per_angle = photos["angle"].value_counts().sort_index()
    per_split = photos["split"].value_counts()
    full_adverts = photos.groupby("advert_id")["angle"].nunique()
    example = (photos[photos["advert_id"] == full_adverts[full_adverts == 8].index[0]]
               .sort_values("angle").groupby("angle").head(1))  # one photo per angle
    return per_brand, per_angle, per_split, example


def funnel(steps, title):
    labels, values, measures = [s[0] for s in steps] + ["Kept"], [s[1] for s in steps] + [0], \
        ["absolute"] + ["relative"] * (len(steps) - 1) + ["total"]
    fig = go.Figure(go.Waterfall(
        x=labels, y=values, measure=measures, text=[f"{v:+,}" if i else f"{v:,}" for i, v in enumerate(values[:-1])]
        + [f"{sum(values):,}"], textposition="outside", textfont=dict(color=TEXT),
        increasing=dict(marker_color=GREEN), decreasing=dict(marker_color=RED), totals=dict(marker_color=GREEN),
        connector=dict(line=dict(color=LINE)),
    ))
    fig.update_yaxes(showgrid=True, gridcolor=LINE, range=[0, steps[0][1] * 1.15])
    fig.update_layout(height=360, showlegend=False)
    with panel(title):
        chart(fig)


st.title("Data")
st.caption("Every photo comes from DVM-CAR, a public dataset of 1.45 million photos from about 250,000 UK car "
           "adverts (2000–2021). Each photo's make, model, year and angle are in its file name and in "
           "`Image_table.csv`.")

path = MODELS_DIR / "features" / "photo_index.csv"
per_brand, per_angle, per_split, example = photo_counts(path.stat().st_mtime if path.exists() else 0)

st.subheader("Two sets of photos")
left, right = st.columns(2)
with left:
    st.markdown("**Fronts, for the make.** The dataset's 61,827 photos confirmed to show the front of the car. "
                "Abarth is merged into Fiat and DS into Citroën, which share their fronts.")
    funnel(FRONTS_FUNNEL, "Front photos → 60,126 kept, 33 makes")
    st.caption("Removed: the same photo filed under two makes, duplicate copies, and 28 makes with under "
               "200 photos.")
with right:
    st.markdown("**Every angle, for the model.** All photos of the same 33 makes, from 8 angles, for car models "
                "with at least 300 photos (421 models, 98% of the photos).")
    funnel(ALL_ANGLES_FUNNEL, "All photos → 1,351,819 kept, 421 models")
    st.caption("Filtered: photos that failed the dataset's quality check, makes outside the 33, and "
               "models with under 300 photos. Then photos with two different labels, and duplicate copies.")

with st.expander("How the photos were cleaned"):
    st.markdown(
        "- **Unreadable photos**: every photo was fully decoded; none were broken.\n"
        "- **Duplicates**: adverts often reuse the same photo. Exact copies were found with an MD5 fingerprint "
        "and kept once, so no photo can sit in both the training and the test set.\n"
        "- **Conflicting labels**: dealers reuse stock photos across adverts. One Kia Picanto photo appears 37 "
        "times, filed under Audi, Citroën, Fiat, Kia and Toyota. A photo filed under two labels is dropped.\n"
        "- **Quality check**: the dataset authors flagged 19,809 photos that don't clearly show a car; they're dropped.\n"
        "- **Split by advert**: all photos of one car go to the same set (80% training, 10% validation, 10% test), "
        "so the model is always tested on cars it has never seen."
    )

st.subheader("One car, eight angles")
st.caption(f"Every advert is photographed from up to 8 angles. Here: a {example['brand'].iloc[0]} "
           f"{example['model'].iloc[0]}, as the models see it (background removed, 300×300).")
for column, (_, row) in zip(st.columns(8), example.iterrows()):
    column.image(str(DATA_DIR / "resized_DVM" / row["path"]), caption=ANGLES[row["angle"]])

st.subheader("Photos per make and per angle")
left, right = st.columns([3, 2])
with left, panel("Photos per make, all angles"):
    fig = go.Figure(go.Bar(
        x=per_brand["photos"], y=per_brand.index, orientation="h", marker_color=GREEN,
        customdata=per_brand["models"],
        hovertemplate="<b>%{y}</b><br>%{x:,} photos · %{customdata} models<extra></extra>",
    ))
    fig.update_xaxes(showgrid=True, gridcolor=LINE, title="photos")
    fig.update_layout(height=22 * len(per_brand) + 60, bargap=0.25)
    chart(fig)
with right:
    with panel("Photos per angle"):
        fig = go.Figure(go.Bar(
            x=[ANGLES[a] for a in per_angle.index], y=per_angle.values,
            marker_color=[GREEN if a == 0 else BLUE for a in per_angle.index],
            hovertemplate="%{x}: %{y:,} photos<extra></extra>",
        ))
        fig.update_yaxes(showgrid=True, gridcolor=LINE)
        fig.update_layout(height=300)
        chart(fig)
    st.caption(f"Ford has {per_brand['photos'].max() / per_brand['photos'].min():.0f}× more photos than "
               f"{per_brand.index[0]}: class weights make every make and model count equally in training.")

st.subheader("Split by advert")
cards = st.columns(3)
for card, split, label, share in zip(cards, ["train", "val", "test"], ["Training photos", "Validation photos", "Test photos"],
                                     ["learns from them", "picks the best epoch", "scored once, at the end"]):
    card.metric(label, f"{per_split[split]:,}", delta=share, delta_color="off", delta_arrow="off", border=True)

Overwriting dashboard/data.py


## 8. Brand model

Step 1 of the tool: test accuracy, error and balanced accuracy; the small CNN and EfficientNetB0 compared on validation, with their training curves; accuracy per make; and which makes get confused (a heatmap of the mistakes and the most common mix-ups).

In [9]:
%%writefile dashboard/brand_model.py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

from common import BG, BLUE, GRAPHITE, GREEN, LINE, MODELS_DIR, PANEL, RED, TEXT, chart, load, panel, pct

NOTEBOOK = "car_brand_predictor.ipynb"

test = load(MODELS_DIR / "test_predictions.csv", NOTEBOOK, usecols=["brand", "predicted_brand", "correct"])
small_cnn = load(MODELS_DIR / "small_cnn_history.csv", NOTEBOOK)
effnet = load(MODELS_DIR / "efficientnet_b0_history.csv", NOTEBOOK)

per_brand = test.groupby("brand")["correct"].mean().sort_values()
accuracy, balanced = test["correct"].mean(), per_brand.mean()

st.title("Brand model")
st.caption("Step 1 of the tool: which make is this, from one photo of the front? Two models were trained on the "
           "60,126 front photos; the better one was scored once on test photos it never saw.")

# ---------- test scores ----------
cards = st.columns(4)
cards[0].metric("Test accuracy", pct(accuracy), border=True,
                help=f"Share of the {len(test):,} test photos whose make is right.")
cards[1].metric("Error", pct(1 - accuracy), border=True,
                help=f"{(~test['correct']).sum()} wrong out of {len(test):,} test photos.")
cards[2].metric("Balanced accuracy", pct(balanced), border=True,
                help="Accuracy of each make, averaged: Suzuki counts as much as Audi, even with 44× fewer photos.")
cards[3].metric("Weakest make", f"{per_brand.index[0]}", delta=pct(per_brand.iloc[0]),
                delta_color="off", delta_arrow="off", border=True)

# ---------- models compared ----------
st.subheader("Two models compared")
st.caption("Both were chosen on the validation set; only the winner was scored on the test set.")
left, right = st.columns([2, 3])
with left, panel("Best validation accuracy"):
    models = pd.DataFrame({
        "model": ["Small CNN, from scratch", "EfficientNetB0, pretrained"],
        "accuracy": [small_cnn["val_accuracy"].max(), effnet["val_accuracy"].max()],
        "epochs": [len(small_cnn), len(effnet)],
    })
    fig = go.Figure(go.Bar(
        x=models["accuracy"], y=models["model"], orientation="h", marker_color=[GRAPHITE, GREEN],
        text=[pct(a) for a in models["accuracy"]], textposition="inside", insidetextanchor="end",
        textfont=dict(color=BG, size=14), customdata=models["epochs"],
        hovertemplate="%{y}: %{x:.1%} after %{customdata} epochs<extra></extra>",
    ))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(range=[0.9, 1], tickformat=".0%", showgrid=True, gridcolor=LINE)
    fig.update_layout(height=200, bargap=0.35)
    chart(fig)
    st.caption("The axis starts at 90%: both are good, the pretrained one makes about half as many mistakes.")
with right, panel("Validation accuracy while training"):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=small_cnn["val_accuracy"], name="Small CNN", line=dict(color=GRAPHITE, width=2)))
    fig.add_trace(go.Scatter(y=effnet["val_accuracy"], name="EfficientNetB0", line=dict(color=GREEN, width=3)))
    fig.add_vline(x=2.5, line=dict(color=BLUE, dash="dot"), annotation_text="EfficientNet: fine-tuning starts",
                  annotation_position="bottom right", annotation_font_color=BLUE)
    fig.update_yaxes(range=[0.7, 1.005], tickformat=".0%", showgrid=True, gridcolor=LINE)
    fig.update_xaxes(title="epoch")
    fig.update_layout(height=280, margin=dict(t=40), legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"),
                      hovermode="x unified")
    chart(fig)

with st.expander("What the two models are"):
    st.markdown(
        "- **Small CNN**: 5 convolution blocks (1.6 million weights) trained from random weights on 128×128 photos. "
        "It learned everything from our 48,000 training photos.\n"
        "- **EfficientNetB0**: a network pretrained on ImageNet (1.3 million everyday photos), at 224×224. We replaced "
        "its last layer with our 33 makes, trained that layer for 3 epochs, then fine-tuned the network's top blocks "
        "(5 to 7) with a 10× lower learning rate.\n"
        "- **Both**: random shifts, zoom, rotation, contrast and brightness while training (never a mirror image, "
        "which would flip the badges); class weights so rare makes count as much as common ones; early stopping on "
        "the validation loss."
    )

# ---------- per make ----------
st.subheader("Accuracy per make")
with panel("Test accuracy per make"):
    fig = go.Figure(go.Bar(
        x=per_brand.values, y=per_brand.index, orientation="h",
        marker_color=[RED if a < 0.97 else GREEN for a in per_brand.values],
        text=[pct(a) for a in per_brand.values], textposition="outside", textfont=dict(color=TEXT),
        hovertemplate="%{y}: %{x:.1%}<extra></extra>",
    ))
    fig.update_xaxes(range=[0.9, 1.02], tickformat=".0%", showgrid=True, gridcolor=LINE)
    fig.update_layout(height=20 * len(per_brand) + 60, bargap=0.25)
    chart(fig)
    st.caption("Red: under 97%. The axis starts at 90%.")

# ---------- confusion ----------
st.subheader("Which makes get confused")
mistakes = test[~test["correct"]]
left, right = st.columns([3, 2])
with left, panel("Mistakes: true make vs predicted make"):
    brands = sorted(test["brand"].unique())
    counts = pd.crosstab(mistakes["brand"], mistakes["predicted_brand"]).reindex(index=brands, columns=brands, fill_value=0)
    fig = go.Figure(go.Heatmap(
        z=counts.replace(0, np.nan).values, x=brands, y=brands, colorscale=[[0, "#3a1d16"], [1, RED]],
        hovertemplate="true %{y} → predicted %{x}: %{z} photos<extra></extra>", showscale=False, xgap=1, ygap=1,
        hoverongaps=False,
    ))
    fig.update_yaxes(autorange="reversed", title="true make", tickmode="array", tickvals=brands, showgrid=False)
    fig.update_xaxes(title="predicted make", tickangle=-60, tickmode="array", tickvals=brands)
    fig.update_layout(height=640, plot_bgcolor=PANEL)
    chart(fig)
with right, panel("Most common mix-ups"):
    top = mistakes.groupby(["brand", "predicted_brand"]).size().sort_values(ascending=False).head(12).reset_index()
    top.columns = ["True make", "Predicted as", "Photos"]
    st.dataframe(top, hide_index=True)
    st.caption(f"Only {len(mistakes)} of {len(test):,} test photos are wrong, and no pair of makes is confused "
               f"more than {top['Photos'].max()} times.")

Overwriting dashboard/brand_model.py


## 9. Car models

Step 2 of the tool: the full pipeline's accuracy; accuracy per make (one photo vs all the car's photos), per angle and per number of photos; fine-tuning (validation curves while it runs, then the before/after test scores); and a confusion heatmap for any make, with its most common mix-ups.

In [10]:
%%writefile dashboard/car_models.py
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

from common import AMBER, BG, BLUE, GRAPHITE, GREEN, LINE, MODELS_DIR, PANEL, RED, TEXT, chart, load, panel, pct

NOTEBOOK = "car_model_predictor.ipynb"
FINETUNE_NOTEBOOK = "car_model_finetune.ipynb"
HEADS_DIR = MODELS_DIR / "model_heads"
FINETUNED_DIR = MODELS_DIR / "model_finetuned"
# DVM-CAR's predicted viewpoint, in degrees around the car
ANGLES = {0: "front (0°)", 45: "45°", 90: "side (90°)", 135: "135°", 180: "rear (180°)",
          225: "225°", 270: "side (270°)", 315: "315°"}

summary = load(HEADS_DIR / "summary.csv", NOTEBOOK)
test = load(HEADS_DIR / "test_predictions.csv", NOTEBOOK,
            usecols=["brand", "model", "angle", "advert_id", "predicted_model", "correct"])
pipeline = load(HEADS_DIR / "pipeline_test.csv", NOTEBOOK)

car_accuracy = np.average(summary["car accuracy"], weights=summary["test cars"])

st.title("Car models")
st.caption("Step 2 of the tool: once the make is known, which of its models is it? Each make has its own "
           "classifier, trained on photos from every angle. Scores are on test cars neither step ever saw.")

# ---------- headline ----------
cards = st.columns(4)
cards[0].metric("Make and model right", pct(pipeline["both correct"].mean()), border=True,
                help=f"The whole tool on {len(pipeline):,} test cars: make from the front photo, model from all "
                     f"of the car's photos ({pipeline['photos'].mean():.1f} on average).")
cards[1].metric("Error", pct(1 - pipeline["both correct"].mean()), border=True,
                help="Test cars where the make or the model is wrong.")
cards[2].metric("Model right, per car", pct(car_accuracy), border=True,
                help="Make given, all photos of a car averaged: the second step on its own.")
cards[3].metric("Model right, one photo", pct(test["correct"].mean()), border=True,
                help="A single photo from any angle, make given. Some angles say much less than others.")

with st.expander("How it works, and why it's cheap"):
    st.markdown(
        f"- **Features, once**: the brand model's EfficientNet turns each of the 1,351,819 photos into 1,280 "
        f"numbers describing it (about 90 minutes on the Mac's GPU, done once and saved).\n"
        f"- **One small classifier per make**: {len(summary)} small networks (one hidden layer) learn the models "
        f"of their make from those numbers: from a few seconds to a minute each, instead of days for 33 full networks.\n"
        f"- **Fine-tuning for the weakest makes**: makes under 90% per car get the EfficientNet itself re-trained "
        f"on their own photos (`{FINETUNE_NOTEBOOK}`).\n"
        f"- **In the tool**: the makes with at least 5% probability from the front photo (up to 3) run their "
        f"classifier on every photo; the photos are averaged, and each (make, model) scores "
        f"P(make) × P(model | make)."
    )

# ---------- per make ----------
st.subheader("Accuracy per make")
with panel("Model accuracy per make, test"):
    ordered = summary.sort_values("car accuracy")
    fig = go.Figure()
    fig.add_trace(go.Bar(x=ordered["photo accuracy"], y=ordered["brand"], orientation="h", name="one photo",
                         marker_color=GRAPHITE, hovertemplate="%{y}, one photo: %{x:.1%}<extra></extra>"))
    fig.add_trace(go.Bar(x=ordered["car accuracy"], y=ordered["brand"], orientation="h", name="per car (all photos)",
                         marker_color=GREEN, customdata=ordered[["models", "test cars"]],
                         hovertemplate="%{y}, per car: %{x:.1%}<br>%{customdata[0]} models · "
                                       "%{customdata[1]:,} test cars<extra></extra>"))
    fig.update_xaxes(range=[0, 1], tickformat=".0%", showgrid=True, gridcolor=LINE)
    fig.update_layout(height=26 * len(ordered) + 80, barmode="group", bargap=0.25, bargroupgap=0.05,
                      legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
    chart(fig)
    st.caption("Makes with many look-alike models are hardest: Audi has 33 models, including A3, S3 and RS3.")

left, right = st.columns(2)
with left, panel("One photo: accuracy per angle"):
    per_angle = test.groupby("angle")["correct"].mean()
    fig = go.Figure(go.Bar(
        x=[ANGLES[a] for a in per_angle.index], y=per_angle.values,
        marker_color=[GREEN if a == per_angle.idxmax() else BLUE for a in per_angle.index],
        text=[pct(a, 0) for a in per_angle.values], textposition="outside", textfont=dict(color=TEXT),
        hovertemplate="%{x}: %{y:.1%}<extra></extra>",
    ))
    fig.update_yaxes(range=[0, 1], tickformat=".0%", showgrid=True, gridcolor=LINE)
    fig.update_layout(height=300)
    chart(fig)
with right, panel("More photos, surer answer"):
    by_photos = pipeline.groupby(pipeline["photos"].clip(upper=6))["both correct"].agg(["mean", "size"])
    fig = go.Figure(go.Bar(
        x=[f"{n}+" if n == 6 else str(n) for n in by_photos.index], y=by_photos["mean"], marker_color=GREEN,
        text=[pct(a, 0) for a in by_photos["mean"]], textposition="outside", textfont=dict(color=TEXT),
        customdata=by_photos["size"], hovertemplate="%{x} photos: %{y:.1%} (%{customdata:,} cars)<extra></extra>",
    ))
    fig.update_xaxes(title="photos of the car")
    fig.update_yaxes(range=[0, 1.05], tickformat=".0%", showgrid=True, gridcolor=LINE)
    fig.update_layout(height=300)
    chart(fig)

# ---------- fine-tuning ----------
st.subheader("Fine-tuning the weakest makes")
comparison_path = FINETUNED_DIR / "comparison.csv"
logs = {p.name.removesuffix("_log.csv"): p for p in sorted(FINETUNED_DIR.glob("*_log.csv")) if p.stat().st_size > 0}
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path).sort_values("car accuracy before")
    with panel("Per car, test: before and after fine-tuning"):
        fig = go.Figure()
        fig.add_trace(go.Bar(x=comparison["car accuracy before"], y=comparison["brand"], orientation="h",
                             name="frozen features", marker_color=GRAPHITE,
                             hovertemplate="%{y}, before: %{x:.1%}<extra></extra>"))
        fig.add_trace(go.Bar(x=comparison["car accuracy after"], y=comparison["brand"], orientation="h",
                             name="fine-tuned", marker_color=GREEN,
                             hovertemplate="%{y}, after: %{x:.1%}<extra></extra>"))
        fig.update_xaxes(range=[0, 1], tickformat=".0%", showgrid=True, gridcolor=LINE)
        fig.update_layout(height=60 * len(comparison) + 80, barmode="group",
                          legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
        chart(fig)
elif logs:
    st.info(f"Fine-tuning is still running (`{FINETUNE_NOTEBOOK}`). The before/after test scores appear here "
            f"when it's done; meanwhile, validation accuracy per epoch:", icon=":material/hourglass_top:")
    with panel("Validation accuracy per photo, per epoch"):
        fig = go.Figure()
        for (brand, path), colour in zip(logs.items(), [GREEN, BLUE, AMBER, RED, GRAPHITE, TEXT]):
            log = pd.read_csv(path)
            fig.add_trace(go.Scatter(x=log["epoch"] + 1, y=log["val_accuracy"], name=brand, mode="lines+markers",
                                     line=dict(color=colour, width=2)))
        fig.update_xaxes(title="epoch", dtick=1)
        fig.update_yaxes(tickformat=".0%", showgrid=True, gridcolor=LINE)
        fig.update_layout(height=300, legend=dict(orientation="h", y=1.02, x=0, yanchor="bottom"))
        chart(fig)
else:
    st.caption(f"Not run yet: `{FINETUNE_NOTEBOOK}`.")

# ---------- confusion ----------
st.subheader("Which models get confused")
brands = summary.sort_values("car accuracy")["brand"].tolist()
brand = st.selectbox("Make", brands, index=0, help="Sorted from the least to the most accurate.")
part = test[test["brand"] == brand]
models = sorted(part["model"].unique())
left, right = st.columns([3, 2])
with left, panel(f"{brand}: true model vs predicted model, one photo", key="confusion"):
    matrix = pd.crosstab(part["model"], part["predicted_model"], normalize="index").reindex(
        index=models, columns=models, fill_value=0)
    fig = go.Figure(go.Heatmap(
        z=matrix.values, x=models, y=models, zmin=0, zmax=1, xgap=1, ygap=1,
        colorscale=[[0, PANEL], [0.15, "#123d24"], [1, GREEN]], colorbar=dict(tickformat=".0%", outlinewidth=0),
        hovertemplate="true %{y} → predicted %{x}: %{z:.0%} of its photos<extra></extra>",
    ))
    fig.update_yaxes(autorange="reversed", title="true model", tickmode="array", tickvals=models, showgrid=False)
    fig.update_xaxes(title="predicted model", tickangle=-60, tickmode="array", tickvals=models)
    fig.update_layout(height=max(360, 22 * len(models) + 160))
    chart(fig)
with right, panel(f"{brand}: most common mix-ups", key="mixups"):
    wrong = part[~part["correct"]]
    top = wrong.groupby(["model", "predicted_model"]).size().sort_values(ascending=False).head(12)
    share = top / part.groupby("model").size().reindex(top.index.get_level_values(0)).values
    table = pd.DataFrame({"True model": top.index.get_level_values(0), "Predicted as": top.index.get_level_values(1),
                          "Photos": top.values, "Share of its photos": share.values})
    st.dataframe(table, hide_index=True,
                 column_config={"Share of its photos": st.column_config.ProgressColumn(format="percent", min_value=0, max_value=1)})
    row = summary.set_index("brand").loc[brand]
    st.caption(f"{brand}: {int(row['models'])} models, {pct(row['photo accuracy'])} per photo, "
               f"{pct(row['car accuracy'])} per car (test, frozen features).")

Overwriting dashboard/car_models.py


## 10. Identify

The tool: upload one or more photos of the same car, mark the front one, and get the make and the model. Shows the verdict, what the models see (each crop with its own best guess) and the top 5 models and makes. Nothing runs until a front photo is marked.

In [11]:
%%writefile dashboard/identify.py
import hashlib

import plotly.graph_objects as go
import streamlit as st

import recognition as rec
from common import GRAPHITE, GREEN, LINE, RED, SUBTEXT, TEXT, chart, panel, pct

UNSURE_BELOW = 0.5   # top probability under this → "not sure"
PER_ROW = 6

st.title("Identify a car")
st.caption("Upload one or more photos of the same car, from any angle, and mark the one that shows the front. "
           "The front gives the make; every photo helps name the model.")

uploads = st.file_uploader("Photos of one car", type=["jpg", "jpeg", "png", "webp"], accept_multiple_files=True)
if not uploads:
    st.info("Best results: one car per photo, clearly visible and not cut off, with at least one photo straight "
            "from the front. Photos from the sides and the back make the model surer.", icon=":material/photo_camera:")
    st.stop()

with st.spinner("Finding the car in each photo..."):
    processed = [rec.crop_car(u.getvalue()) for u in uploads]
names = [f"Photo {i + 1}" for i in range(len(uploads))]


def grid(items):
    """Show (caption, image) pairs, PER_ROW per row."""
    for start in range(0, len(items), PER_ROW):
        for column, (caption, image) in zip(st.columns(PER_ROW), items[start:start + PER_ROW]):
            column.image(image, caption=caption)


with panel("Your photos"):
    grid([(name + ("" if crop is not None else " · no car found"), photo)
          for name, (photo, crop, _) in zip(names, processed)])

front = st.segmented_control(
    "Which photo shows the front of the car?", options=range(len(uploads)), format_func=lambda i: names[i],
    default=0 if len(uploads) == 1 else None,
    key="front_" + hashlib.md5("".join(f"{u.name}{u.size}" for u in uploads).encode()).hexdigest(),
)
if front is None:
    st.info("Mark the front photo above to start: the make is read from the front of the car.",
            icon=":material/touch_app:")
    st.stop()

front_crop = processed[front][1]
if front_crop is None:
    st.error(f"No car found in {names[front]}. The front photo is needed for the make: try one where the car is "
             f"larger and fully in the frame.", icon=":material/error:")
    st.stop()

usable = [i for i, (_, crop, _) in enumerate(processed) if crop is not None]
with st.spinner("Naming the make and the model..."):
    brand_ranking = rec.predict_brand(front_crop)
    has_models = rec.car_models_available()
    if has_models:
        model_ranking, per_photo, finetuned = rec.predict_car_model(brand_ranking, [processed[i][1] for i in usable])

p_brands = dict(brand_ranking)

# ---------- result ----------
if has_models:
    brand, model, score = model_ranking[0]
    p_model = score / p_brands[brand]
    sure = p_brands[brand] >= UNSURE_BELOW and p_model >= UNSURE_BELOW
    cards = st.columns(4)
    cards[0].metric("Make", brand, delta=f"{pct(p_brands[brand], 0)} from the front", delta_arrow="off",
                    delta_color="normal" if p_brands[brand] >= UNSURE_BELOW else "inverse", border=True)
    cards[1].metric("Model", model, delta=f"{pct(p_model, 0)} given the make", delta_arrow="off",
                    delta_color="normal" if p_model >= UNSURE_BELOW else "inverse", border=True)
    cards[2].metric("Overall", pct(score, 0), border=True, help="P(make) × P(model | make).")
    cards[3].metric("Photos used", f"{len(usable)} of {len(uploads)}", border=True,
                    help="Photos where no car was found are skipped.")
    colour = GREEN if sure else RED
    verdict = (f"This looks like a <b style='color:{colour}'>{brand} {model}</b>."
               if sure else f"Not sure. Best guess: <b style='color:{colour}'>{brand} {model}</b>, but the "
                            f"{'make' if p_brands[brand] < UNSURE_BELOW else 'model'} is uncertain. "
                            f"More photos from other sides usually help.")
else:
    brand, p_brand = brand_ranking[0]
    sure = p_brand >= UNSURE_BELOW
    colour = GREEN if sure else RED
    st.columns(4)[0].metric("Make", brand, delta=pct(p_brand, 0), delta_arrow="off", border=True)
    verdict = (f"The make is <b style='color:{colour}'>{brand}</b>. Car models aren't available yet: "
               f"run <code>car_model_predictor.ipynb</code>.")
st.markdown(f"<div style='border-left:4px solid {colour}; padding:0.75rem 1rem; margin:0.5rem 0 1.5rem; "
            f"font-size:1.15rem; color:{TEXT}'>{verdict}</div>", unsafe_allow_html=True)

# ---------- what the models see ----------
with panel("What the models see", key="crops"):
    st.caption("Each car cut out of its background, like the training photos"
               + (f", with {brand}'s best model for that photo alone." if has_models else "."))
    items = []
    for i, name in enumerate(names):
        crop = processed[i][1]
        if crop is None:
            continue
        caption = name + (" · front" if i == front else "")
        if has_models and brand in per_photo:
            photo_model, p = per_photo[brand][usable.index(i)]
            caption += f" · {photo_model} {pct(p, 0)}"
        items.append((caption, crop))
    grid(items)


def bars(items, title, key):
    labels, values = [label for label, _ in items], [p for _, p in items]
    fig = go.Figure(go.Bar(
        x=values, y=labels, orientation="h", marker_color=[GREEN] + [GRAPHITE] * (len(items) - 1),
        text=[pct(v) for v in values], textposition="outside", textfont=dict(color=TEXT),
        hovertemplate="%{y}: %{x:.1%}<extra></extra>",
    ))
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(range=[0, 1.15], tickformat=".0%", showgrid=True, gridcolor=LINE)
    fig.update_layout(height=250, bargap=0.35)
    with panel(title, key=key):
        chart(fig)


left, right = st.columns(2)
with left:
    if has_models:
        bars([(f"{b} {m}", s) for b, m, s in model_ranking[:5]], "Top 5 models", "top_models")
        used = [f"{b}: {'fine-tuned' if f else 'standard'} classifier" for b, f in finetuned.items()]
        st.caption("Makes checked: " + ", ".join(used) + ".")
with right:
    bars(brand_ranking[:5], f"Top 5 makes, from {names[front].lower()}", "top_brands")

st.caption(f"<span style='color:{SUBTEXT}'>The tool knows 33 makes and 421 models, mostly UK cars from 2000–2021; "
           f"anything else is reported as the closest car it knows. Only the biggest car in each photo is used.</span>",
           unsafe_allow_html=True)

Overwriting dashboard/identify.py


## 11. Check every page

`AppTest` runs each page without a browser, the way Streamlit would, and reports any error. The Identify page stops at the photo upload, which is expected here (there's no photo to upload).

In [12]:
from streamlit.testing.v1 import AppTest

PAGES = ["home", "data", "brand_model", "car_models", "identify"]
for page in PAGES:
    # absolute path: AppTest resolves relative paths from where it is called, not from the project
    test = AppTest.from_file(str(Path(f"dashboard/{page}.py").resolve()), default_timeout=120).run()
    errors = [e.value for e in test.exception]
    numbers = [m.value for m in test.metric]
    print(f"{page:12} {'ERROR: ' + errors[0][:200] if errors else 'ok'}   {' · '.join(numbers)}")

2026-09-30 09:55:41.874 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.

2026-09-30 09:55:44.358 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


home         ok   1,351,819 · 99.4% · 89.5%


2026-09-30 09:55:47.079 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


data         ok   1,026,733 · 161,716 · 163,370


2026-09-30 09:55:47.418 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


brand_model  ok   99.4% · 0.6% · 98.9% · Mitsubishi


2026-09-30 09:55:48.118 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


car_models   ok   89.5% · 10.5% · 91.4% · 78.5%

identify     ok   


## 12. Start the app

Starts Streamlit in the background and opens it at http://localhost:8501. If it's already running (for example from a terminal), it just says so: refresh the browser to see changes.

From a terminal instead: `.venv/bin/streamlit run dashboard/app.py`

In [13]:
import socket
import subprocess
import sys

PORT = 8501


def running(port):
    with socket.socket() as s:
        return s.connect_ex(("localhost", port)) == 0


if running(PORT):
    print(f"Already running: http://localhost:{PORT}")
else:
    streamlit = Path(sys.executable).parent / "streamlit"
    app = subprocess.Popen([str(streamlit), "run", "dashboard/app.py", "--server.port", str(PORT)])
    print(f"Started: http://localhost:{PORT}  (stop it with the next cell)")

Already running: http://localhost:8501


In [14]:
# Stop the app started by the cell above
if "app" in globals() and app.poll() is None:
    app.terminate()
    print("Stopped")
else:
    print("Not started from this notebook")

Not started from this notebook
